# SARIMA & Prophet

先把统计模型跑一遍当baseline。训练用2015-2017，2018拿来测试。

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.arima_model import SarimaForecaster, naive_forecast, seasonal_naive
from src.models.prophet_model import ProphetForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, save_figure

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)
print(f'Train: {len(train)}m, Test: {len(test)}m')

## Naive baselines

最简单的两个：重复上个月 / 重复去年同月。

In [ ]:
naive_pred = naive_forecast(train, test)
snaive_pred = seasonal_naive(train, test)

results = []
results.append(evaluate_forecast(test, naive_pred, 'Naive'))
results.append(evaluate_forecast(test, snaive_pred, 'Seasonal Naive'))

for r in results:
    print(f"{r['model']:20s} RMSE={r['RMSE']:>10}  MAE={r['MAE']:>10}  MAPE={r['MAPE']}%")

In [ ]:
fig = plot_forecast(train, test, snaive_pred, 'Seasonal Naive')
plt.show()

Seasonal Naive RMSE只有Naive的一半左右，说明这个数据的季节性pattern确实很dominant。

## SARIMA

d=1, m=12，让auto_arima搜一下。

In [ ]:
sarima = SarimaForecaster()
sarima.auto_fit(train, seasonal=True, m=12)

In [ ]:
sarima_pred = sarima.predict(steps=len(test))
sarima_pred.index = test.index

results.append(evaluate_forecast(test, sarima_pred, 'SARIMA'))
print(results[-1])

In [ ]:
fig = plot_forecast(train, test, sarima_pred, 'SARIMA')
save_figure(fig, 'sarima_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
print(sarima.get_summary())

SARIMA的RMSE比Seasonal Naive还高。预测曲线方向是对的，但有几个月偏差比较大，可能是36个月数据不够参数估计不稳定。

## Prophet

In [ ]:
prophet = ProphetForecaster(
    yearly_seasonality=True,
    changepoint_prior_scale=0.05
)
prophet.fit(train)

In [ ]:
prophet_pred = prophet.predict(periods=len(test))
prophet_pred.index = test.index

results.append(evaluate_forecast(test, prophet_pred.values, 'Prophet'))
print(results[-1])

In [ ]:
fig = plot_forecast(train, test, prophet_pred.values, 'Prophet')
save_figure(fig, 'prophet_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
# 看看component拆解
fc = prophet.get_components(periods=len(test))
prophet.model.plot_components(fc)
plt.tight_layout()
plt.show()

Prophet也没赢Seasonal Naive。可能是changepoint_prior_scale太保守了trend拟合偏平，但数据量少也不敢调太大。

## 对比

In [ ]:
compare_models(results)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train.values, color='gray', alpha=0.5, label='Train')
ax.plot(test.index, test.values, color='black', lw=2, label='Actual')
ax.plot(test.index, sarima_pred.values, '--', label='SARIMA')
ax.plot(test.index, prophet_pred.values, '--', label='Prophet')
ax.plot(test.index, snaive_pred.values, ':', label='Seasonal Naive')
ax.legend()
ax.set_title('All Forecasts vs Actual')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Seasonal Naive赢了。36个月数据有限，复杂模型学不到什么额外的东西。